#Incremental load

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/jilmedavidesco@gmail.com/pipeline_project/1_code/1_setup/2_utilities

In [0]:
print(bronze_schema_parent, silver_schema_parent, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "atlone_enterprise", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")
dbutils.widgets.text("company", "parent", "Company")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
company = dbutils.widgets.get("company")


base_path = f's3://bucket-storage-company-parents-child/{company }/{data_source}' 
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)

# define the tables
bronze_table = f"{catalog}.{bronze_schema_parent}.{data_source}"
silver_table = f"{catalog}.{silver_schema_parent}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.parent_fact_{data_source}"

###Bronze

In [0]:
# 1. Verificar si hay archivos CSV en landing
archivos_landing = dbutils.fs.ls(landing_path)
hay_archivos_csv = any(file.name.endswith('.csv') for file in archivos_landing)

if hay_archivos_csv:
    # 2. Leer los datos si existen
    df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv") \
        .withColumn("read_timestamp", F.current_timestamp()) \
        .select("*", "_metadata.file_name", "_metadata.file_size")
    
    print("Total Rows a insertar: ", df.count())
    
    # 3. Escribir en la capa Bronze
    (df.write
        .format("delta")
        .option("delta.enableChangeDataFeed", "true")
        .mode("append")
        .saveAsTable(bronze_table)
    )
    print("Escritura en Bronze completada.")
else:
    # 4. Si no hay archivos, salir limpiamente
    print("Carpeta landing vacía. Proceso finalizado sin cambios.")

### Staging table to process just the arrived incremenal data

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema_parent}.staging_{data_source}")

### Moving files from source to processed directory

In [0]:
files = dbutils.fs.ls(landing_path)
print(files)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

###Silver

In [0]:
df_orders = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema_parent}.staging_{data_source};")
df_orders.show(2)

**Transformations**

In [0]:
null_counts = df_orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_orders.columns
])
display(null_counts)

In [0]:
df_orders.printSchema()

In [0]:
# 1. Keep only rows where sold_quantity is present
df_orders = df_orders.filter(F.col("sold_quantity").isNotNull())

# 2. Parse date using multiple possible formats
df_orders = df_orders.withColumn(
    "date",
    F.coalesce(
        F.try_to_date("date", "yyyy-MM-dd"),
        F.try_to_date("date", "yyyy/MM/dd"),
        F.try_to_date("date", "dd-MM-yyyy"),
        F.try_to_date("date", "dd/MM/yyyy"),
        F.try_to_date("date", "MMMM dd, yyyy")
    )
)

# 3. drop duplicate 
df_orders = df_orders.dropDuplicates(["date", "product_code", "customer_code", "sold_quantity"])

# 4.cambia los negativos por psoitivos
df_orders = df_orders.withColumn("sold_quantity", F.abs(F.col("sold_quantity")).cast("integer"))

# 5. cambiar string
df_orders = df_orders.withColumn("customer_code", F.col("customer_code").cast("string"))

In [0]:
df_orders.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).show()

left semi join with products

In [0]:
# Integridad Referencial (LEFT SEMI JOIN) contra Clientes y Productos
df_silver_customers = spark.table(f"{catalog}.{silver_schema_parent}.customers")
df_silver_products = spark.table(f"{catalog}.{silver_schema_parent}.products")

df_silver_valid = (df_orders
    .join(df_silver_customers, on="customer_code", how="leftsemi")
    .join(df_silver_products, on="product_code", how="leftsemi")
)

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_silver_valid.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)

    silver_delta.alias("silver_p").merge(df_silver_valid.alias("bronze_p"),"silver_p.date = bronze_p.date AND silver_p.product_code = bronze_p.product_code AND silver_p.customer_code = bronze_p.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    

### Staging table to process just the arrived incremenal data

In [0]:
# stagging for incremental data

df_silver_valid.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema_parent}.staging_{data_source}")

###Gold

In [0]:
df_gold = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {catalog}.{silver_schema_parent}.staging_{data_source};")
df_gold.show(5)

In [0]:
df_gold.count()

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("Creating table")
    df_gold.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)

    gold_delta.alias("source").merge(df_gold.alias("gold"),"source.date = gold.date AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    

## Cleanup

In [0]:
%sql
DROP TABLE IF EXISTS atlone_enterprise.bronze_parent.staging_orders;

In [0]:
%sql
DROP TABLE IF EXISTS atlone_enterprise.silver_parent.staging_orders;